# cuslide vs cuslide2 Benchmark

This notebook compares the **original cuslide** plugin against the **cuslide2**
plugin (nvImageCodec-backed) across several workloads:

| Benchmark | What it measures |
|-----------|------------------|
| Single-tile read | Latency for one 256×256 tile |
| Region read (CPU) | Throughput for a large ROI to host memory |
| Region read (GPU) | Throughput for a large ROI to device memory |
| Multi-tile sequential | N sequential 256×256 reads |
| Batch decode (GPU) | Batched multi-tile decode to GPU |
| Tile-level caching (cuslide2 only) | Cache hit/miss performance |

**Requirements:**
- Both `cucim.kit.cuslide@26.04.00.so` and `cucim.kit.cuslide2@26.04.00.so` built
- Test image at `/tmp/CMU-1-Small-Region.svs`
- CUDA-capable GPU + driver

> **Note:** Each plugin switch requires a kernel restart because cuCIM loads
> plugins once at process startup. This notebook uses `subprocess` to run
> each benchmark in an isolated process.

In [ ]:
import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuration

In [ ]:
REPO_ROOT = Path(os.path.abspath(".."))
CUSLIDE_LIB = REPO_ROOT / "cpp" / "plugins" / "cucim.kit.cuslide" / "build-release" / "lib"
CUSLIDE2_LIB = REPO_ROOT / "cpp" / "plugins" / "cucim.kit.cuslide2" / "build-release" / "lib"
LIBCUCIM_DIR = REPO_ROOT / "install" / "lib"

PLUGIN_VERSION = "26.04.00"
CUSLIDE_SO = f"cucim.kit.cuslide@{PLUGIN_VERSION}.so"
CUSLIDE2_SO = f"cucim.kit.cuslide2@{PLUGIN_VERSION}.so"

TEST_IMAGE = "/tmp/CMU-1-Small-Region.svs"

N_WARMUP = 2
N_REPEATS = 10

assert Path(TEST_IMAGE).exists(), f"Test image not found: {TEST_IMAGE}"
assert (CUSLIDE_LIB / CUSLIDE_SO).exists(), f"cuslide plugin not found: {CUSLIDE_LIB / CUSLIDE_SO}"
assert (CUSLIDE2_LIB / CUSLIDE2_SO).exists(), f"cuslide2 plugin not found: {CUSLIDE2_LIB / CUSLIDE2_SO}"
print("All prerequisites OK")
print(f"  cuslide:  {CUSLIDE_LIB / CUSLIDE_SO}")
print(f"  cuslide2: {CUSLIDE2_LIB / CUSLIDE2_SO}")
print(f"  image:    {TEST_IMAGE}")

## Benchmark Runner

Since cuCIM loads plugins once per process, we run each benchmark in a
subprocess with the appropriate environment variables.

In [ ]:
BENCHMARK_SCRIPT = textwrap.dedent(r'''
import json
import os
import sys
import time

import numpy as np

plugin_root = os.environ["_BENCH_PLUGIN_ROOT"]
plugin_name = os.environ["_BENCH_PLUGIN_NAME"]
image_path = os.environ["_BENCH_IMAGE"]
n_warmup = int(os.environ.get("_BENCH_WARMUP", "2"))
n_repeats = int(os.environ.get("_BENCH_REPEATS", "10"))

os.environ["CUCIM_PLUGINS"] = plugin_name
os.environ.pop("CUCIM_CONFIG_PATH", None)
os.environ.pop("ENABLE_CUSLIDE2", None)

from cucim.clara import CuImage, _set_plugin_root
_set_plugin_root(plugin_root)

img = CuImage(image_path)
dims = img.resolutions["level_dimensions"]
w, h = dims[0]
tile_sizes = img.resolutions.get("level_tile_sizes", ())
tile_w = tile_sizes[0][0] if tile_sizes and tile_sizes[0][0] > 0 else 256
tile_h = tile_sizes[0][1] if tile_sizes and tile_sizes[0][1] > 0 else 256

results = {}

def bench(name, fn, warmup=n_warmup, repeats=n_repeats):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    times_ms = [t * 1000 for t in times]
    results[name] = {
        "mean_ms": float(np.mean(times_ms)),
        "std_ms": float(np.std(times_ms)),
        "min_ms": float(np.min(times_ms)),
        "max_ms": float(np.max(times_ms)),
        "median_ms": float(np.median(times_ms)),
        "n": repeats,
    }

# --- 1. Single tile read (CPU) ---
bench("single_tile_cpu", lambda: np.asarray(img.read_region((0, 0), (256, 256), level=0)))

# --- 2. Large region read (CPU) ---
roi_w = min(2048, w)
roi_h = min(2048, h)
bench("region_2k_cpu", lambda: np.asarray(img.read_region((0, 0), (roi_w, roi_h), level=0)))

# --- 3. Large region read (GPU) ---
try:
    import cupy as cp
    def gpu_region():
        r = img.read_region((0, 0), (roi_w, roi_h), level=0, device="cuda")
        cp.cuda.get_current_stream().synchronize()
        return r
    bench("region_2k_gpu", gpu_region)
except Exception as e:
    results["region_2k_gpu"] = {"error": str(e)}

# --- 4. Multi-tile sequential (CPU) ---
tile_locs = []
for y in range(0, min(h, tile_h * 4), tile_h):
    for x in range(0, min(w, tile_w * 4), tile_w):
        tile_locs.append((x, y))
n_tiles = len(tile_locs)

def seq_tiles():
    for loc in tile_locs:
        np.asarray(img.read_region(loc, (tile_w, tile_h), level=0))

bench("sequential_tiles_cpu", seq_tiles)
results["sequential_tiles_cpu"]["n_tiles"] = n_tiles

# --- 5. Batch decode (GPU) ---
try:
    import cupy as cp
    batch_locs = tile_locs[:min(16, n_tiles)]
    def batch_gpu():
        tiles = list(img.read_region(
            location=batch_locs,
            size=(tile_w, tile_h),
            level=0,
            device="cuda",
            batch_size=len(batch_locs),
            num_workers=1,
        ))
        cp.cuda.get_current_stream().synchronize()
        return tiles
    bench("batch_tiles_gpu", batch_gpu)
    results["batch_tiles_gpu"]["batch_size"] = len(batch_locs)
except Exception as e:
    results["batch_tiles_gpu"] = {"error": str(e)}

# --- 6. Tile-level cache (cuslide2 only) ---
if "cuslide2" in plugin_name:
    try:
        CuImage.cache("per_process", memory_capacity=256)
        CuImage.cache().record(True)
        img_cached = CuImage(image_path)

        # Cold read
        t0 = time.perf_counter()
        np.asarray(img_cached.read_region((0, 0), (roi_w, roi_h), level=0))
        cold_ms = (time.perf_counter() - t0) * 1000
        cold_misses = CuImage.cache().miss_count

        # Warm reads
        warm_times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            np.asarray(img_cached.read_region((0, 0), (roi_w, roi_h), level=0))
            warm_times.append((time.perf_counter() - t0) * 1000)
        warm_hits = CuImage.cache().hit_count

        results["cache_cold_ms"] = cold_ms
        results["cache_warm_mean_ms"] = float(np.mean(warm_times))
        results["cache_warm_std_ms"] = float(np.std(warm_times))
        results["cache_speedup"] = cold_ms / float(np.mean(warm_times)) if np.mean(warm_times) > 0 else 0
        results["cache_total_hits"] = warm_hits
        results["cache_total_misses"] = cold_misses
    except Exception as e:
        results["cache_error"] = str(e)

results["_meta"] = {
    "plugin": plugin_name,
    "image": image_path,
    "image_dims": [w, h],
    "tile_dims": [tile_w, tile_h],
}

print(json.dumps(results))
''')


def run_benchmark(plugin_root, plugin_name, image_path, n_warmup=N_WARMUP, n_repeats=N_REPEATS):
    """Run the benchmark script in a subprocess with the given plugin."""
    env = os.environ.copy()
    env["_BENCH_PLUGIN_ROOT"] = str(plugin_root)
    env["_BENCH_PLUGIN_NAME"] = plugin_name
    env["_BENCH_IMAGE"] = str(image_path)
    env["_BENCH_WARMUP"] = str(n_warmup)
    env["_BENCH_REPEATS"] = str(n_repeats)
    env.pop("CUCIM_PLUGINS", None)
    env.pop("CUCIM_CONFIG_PATH", None)
    env.pop("ENABLE_CUSLIDE2", None)

    ld_path = env.get("LD_LIBRARY_PATH", "")
    libcucim = str(LIBCUCIM_DIR)
    if libcucim not in ld_path:
        env["LD_LIBRARY_PATH"] = f"{libcucim}:{ld_path}" if ld_path else libcucim

    result = subprocess.run(
        [sys.executable, "-c", BENCHMARK_SCRIPT],
        capture_output=True,
        text=True,
        env=env,
        timeout=300,
    )

    if result.returncode != 0:
        print(f"STDERR ({plugin_name}):\n{result.stderr}")
        raise RuntimeError(f"Benchmark failed for {plugin_name}")

    # The last line of stdout is the JSON results
    lines = result.stdout.strip().split("\n")
    return json.loads(lines[-1])

print("Benchmark runner ready")

## Run Benchmarks

In [ ]:
print("Running cuslide benchmark...")
cuslide_results = run_benchmark(CUSLIDE_LIB, CUSLIDE_SO, TEST_IMAGE)
print(f"  Done. Image: {cuslide_results['_meta']['image_dims']}, "
      f"tiles: {cuslide_results['_meta']['tile_dims']}")

print("\nRunning cuslide2 benchmark...")
cuslide2_results = run_benchmark(CUSLIDE2_LIB, CUSLIDE2_SO, TEST_IMAGE)
print(f"  Done. Image: {cuslide2_results['_meta']['image_dims']}, "
      f"tiles: {cuslide2_results['_meta']['tile_dims']}")

print("\nBoth benchmarks complete!")

## Results Table

In [ ]:
benchmarks = ["single_tile_cpu", "region_2k_cpu", "region_2k_gpu",
              "sequential_tiles_cpu", "batch_tiles_gpu"]

rows = []
for bm in benchmarks:
    cs = cuslide_results.get(bm, {})
    cs2 = cuslide2_results.get(bm, {})

    if "error" in cs or "error" in cs2:
        rows.append({
            "Benchmark": bm,
            "cuslide (ms)": cs.get("error", f"{cs.get('mean_ms', 0):.2f} +/- {cs.get('std_ms', 0):.2f}"),
            "cuslide2 (ms)": cs2.get("error", f"{cs2.get('mean_ms', 0):.2f} +/- {cs2.get('std_ms', 0):.2f}"),
            "Speedup": "N/A",
        })
        continue

    cs_mean = cs.get("mean_ms", 0)
    cs2_mean = cs2.get("mean_ms", 0)
    speedup = cs_mean / cs2_mean if cs2_mean > 0 else float("inf")

    extra = ""
    if "n_tiles" in cs:
        extra = f" ({cs['n_tiles']} tiles)"
    if "batch_size" in cs:
        extra = f" (batch={cs['batch_size']})"

    rows.append({
        "Benchmark": bm + extra,
        "cuslide (ms)": f"{cs_mean:.2f} +/- {cs.get('std_ms', 0):.2f}",
        "cuslide2 (ms)": f"{cs2_mean:.2f} +/- {cs2.get('std_ms', 0):.2f}",
        "Speedup": f"{speedup:.2f}x",
    })

df = pd.DataFrame(rows)
display(df)

# Cache results (cuslide2 only)
if "cache_cold_ms" in cuslide2_results:
    print(f"\n--- Tile-level Cache (cuslide2 only) ---")
    print(f"  Cold read:  {cuslide2_results['cache_cold_ms']:.2f} ms")
    print(f"  Warm read:  {cuslide2_results['cache_warm_mean_ms']:.2f} "
          f"+/- {cuslide2_results['cache_warm_std_ms']:.2f} ms")
    print(f"  Speedup:    {cuslide2_results['cache_speedup']:.2f}x")
    print(f"  Hits/Misses: {cuslide2_results['cache_total_hits']}/{cuslide2_results['cache_total_misses']}")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart: latency comparison ---
ax = axes[0]
labels = []
cs_means = []
cs_stds = []
cs2_means = []
cs2_stds = []

for bm in benchmarks:
    cs = cuslide_results.get(bm, {})
    cs2 = cuslide2_results.get(bm, {})
    if "error" in cs or "error" in cs2:
        continue
    labels.append(bm.replace("_", "\n"))
    cs_means.append(cs["mean_ms"])
    cs_stds.append(cs["std_ms"])
    cs2_means.append(cs2["mean_ms"])
    cs2_stds.append(cs2["std_ms"])

x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, cs_means, width, yerr=cs_stds, label="cuslide",
               color="#4c72b0", alpha=0.85, capsize=3)
bars2 = ax.bar(x + width/2, cs2_means, width, yerr=cs2_stds, label="cuslide2",
               color="#55a868", alpha=0.85, capsize=3)

ax.set_ylabel("Latency (ms)")
ax.set_title("cuslide vs cuslide2: Latency Comparison")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# --- Cache speedup chart (cuslide2 only) ---
ax2 = axes[1]
if "cache_cold_ms" in cuslide2_results:
    cache_labels = ["Cold read\n(decode + insert)", "Warm read\n(cache hit)"]
    cache_vals = [cuslide2_results["cache_cold_ms"],
                  cuslide2_results["cache_warm_mean_ms"]]
    cache_errs = [0, cuslide2_results["cache_warm_std_ms"]]
    colors = ["#dd8452", "#55a868"]

    bars = ax2.bar(cache_labels, cache_vals, yerr=cache_errs,
                   color=colors, alpha=0.85, capsize=5, width=0.5)
    ax2.set_ylabel("Latency (ms)")
    ax2.set_title(f"cuslide2 Tile Cache: {cuslide2_results['cache_speedup']:.1f}x speedup")
    ax2.grid(axis="y", alpha=0.3)

    for bar, val in zip(bars, cache_vals):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.1f} ms", ha="center", va="bottom", fontsize=9)
else:
    ax2.text(0.5, 0.5, "Cache data not available",
             ha="center", va="center", transform=ax2.transAxes, fontsize=12)
    ax2.set_title("cuslide2 Tile Cache")

plt.tight_layout()
plt.savefig("cuslide_vs_cuslide2_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: cuslide_vs_cuslide2_benchmark.png")

## Speedup Summary

In [ ]:
print("=" * 60)
print("SPEEDUP SUMMARY (cuslide2 / cuslide)")
print("=" * 60)

for bm in benchmarks:
    cs = cuslide_results.get(bm, {})
    cs2 = cuslide2_results.get(bm, {})
    if "error" in cs or "error" in cs2:
        print(f"  {bm:30s}  SKIPPED (error)")
        continue
    speedup = cs["mean_ms"] / cs2["mean_ms"] if cs2["mean_ms"] > 0 else 0
    indicator = "faster" if speedup > 1 else "slower"
    print(f"  {bm:30s}  {speedup:.2f}x ({indicator})")

if "cache_cold_ms" in cuslide2_results:
    print(f"\n  {'tile_cache (cold->warm)':30s}  "
          f"{cuslide2_results['cache_speedup']:.2f}x (cuslide2 only)")

print("=" * 60)

## Raw Results (JSON)

In [ ]:
print("--- cuslide ---")
print(json.dumps(cuslide_results, indent=2))
print("\n--- cuslide2 ---")
print(json.dumps(cuslide2_results, indent=2))